# Árboles de Decisión para Clasificación _(versión detallada)_

_Gini, entropía y comparación con la regresión logística — caso: default crediticio_

**Módulo 1 — Aprendizaje Supervisado** | DSRP Machine Learning Engineering  
**Profesor:** Miguel Arquez

---
> 📖 **Nota:** Esta es la versión detallada del notebook `05_arboles_decision_clasificacion.ipynb`.  
> Incluye teoría ampliada y comentarios línea a línea en cada bloque de código.

![Aprendizaje Supervisado](assets/header.png)

## 1. Intuición — de regresión a clasificación

Los árboles de decisión para clasificación funcionan con la misma estructura que los de regresión: **preguntas binarias** que dividen el espacio en regiones rectangulares. La diferencia está en **qué predice cada hoja**.

### Diferencias clave

| Aspecto | Árbol de Regresión | Árbol de Clasificación |
|---------|-------------------|------------------------|
| **Predicción en cada hoja** | Media de $y$ | Clase mayoritaria |
| **Probabilidades** | No aplica | Frecuencias relativas de cada clase |
| **Criterio de split** | Minimizar varianza (MSE) | Minimizar impureza (Gini o Entropía) |
| **Métrica de evaluación** | MAE, RMSE, R² | Accuracy, Precision, Recall, F1, ROC-AUC |

### ¿Cómo predice?

Para una nueva observación:
1. Recorre el árbol según las reglas (igual que en regresión)
2. Llega a una hoja
3. **Clase predicha:** La clase mayoritaria de las observaciones de entrenamiento en esa hoja
4. **Probabilidades:** La proporción de cada clase en esa hoja

**Ejemplo:** Si una hoja tiene 80 observaciones (60 clase 0, 20 clase 1):
- Predicción: clase 0 (mayoritaria)
- Probabilidades: P(clase 0) = 60/80 = 0.75, P(clase 1) = 20/80 = 0.25

> 💡 **Ventaja:** A diferencia de la regresión logística (que asume una relación lineal entre log-odds y features), el árbol puede capturar **fronteras de decisión complejas** (no lineales, con interacciones).

## 2. ¿Qué hace un "buen" split? — pureza e impureza

En clasificación, queremos splits que dejen los hijos lo más **"puros"** posible: idealmente, todas las observaciones de un hijo de la misma clase.

### Concepto de pureza

- **Nodo puro:** Todas las observaciones son de la misma clase → fácil predecir, bajo error
- **Nodo impuro:** Mezcla de clases → difícil predecir, alto error

### Índices de impureza

Para medir cuán impuro es un nodo se usan dos índices:

#### 1. Índice de Gini (default en scikit-learn)

$$
G = 1 - \sum_{k=1}^{K} p_k^{2}
$$

donde $p_k$ es la proporción de la clase $k$ en el nodo.

**Interpretación:** Probabilidad de clasificar incorrectamente una observación si la asignamos aleatoriamente según la distribución del nodo.

**Ejemplo (binario):**
- Nodo con 100% clase 0: $G = 1 - 1^2 = 0$ (puro)
- Nodo con 50% clase 0, 50% clase 1: $G = 1 - 0.5^2 - 0.5^2 = 0.5$ (máxima impureza)
- Nodo con 80% clase 0, 20% clase 1: $G = 1 - 0.8^2 - 0.2^2 = 0.36$

#### 2. Entropía (de teoría de la información)

$$
H = -\sum_{k=1}^{K} p_k \log_2 p_k
$$

**Interpretación:** Cantidad de "incertidumbre" o "desorden" en el nodo. Mide cuántos bits necesitas para codificar la clase de una observación.

**Ejemplo (binario):**
- Nodo con 100% clase 0: $H = 0$ (sin incertidumbre)
- Nodo con 50% clase 0, 50% clase 1: $H = 1$ (máxima incertidumbre)
- Nodo con 80% clase 0, 20% clase 1: $H = 0.72$

### Ganancia de información

El árbol elige el split que **maximiza la ganancia de información**:

$$
\text{Ganancia} = \text{Impureza}_{\text{padre}} - \left( \frac{n_{\text{izq}}}{n} \cdot \text{Impureza}_{\text{izq}} + \frac{n_{\text{der}}}{n} \cdot \text{Impureza}_{\text{der}} \right)
$$

Es decir, busca el split que más reduce la impureza ponderada por el tamaño de los hijos.

> 📌 **En la práctica:** Gini y Entropía dan resultados muy similares. Gini es ligeramente más rápido de calcular (no tiene logaritmo), por eso es el default.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Generamos un rango de proporciones de la clase 1 (de 0.001 a 0.999)
# Evitamos 0 y 1 exactos porque log(0) no está definido
p = np.linspace(0.001, 0.999, 200)

# Calculamos el índice de Gini para cada proporción
# En un problema binario: G = 1 - p² - (1-p)²
gini = 1 - p**2 - (1 - p)**2

# Calculamos la entropía para cada proporción
# En un problema binario: H = -p·log₂(p) - (1-p)·log₂(1-p)
ent = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

# --- Gráfico ---
fig, ax = plt.subplots(figsize=(7, 4))

# Línea azul: Índice de Gini
ax.plot(p, gini, label='Gini', lw=2, color='blue')

# Línea naranja: Entropía
ax.plot(p, ent, label='Entropía', lw=2, color='orange')

ax.set_xlabel('p (proporción de la clase 1)')
ax.set_ylabel('impureza')
ax.set_title('Gini y entropía en un nodo binario')
ax.legend()
ax.grid(True)
plt.show()

# OBSERVACIONES:
# 1. Ambas curvas tienen forma de campana (máximo en p=0.5)
# 2. Ambas valen 0 en los extremos (nodos puros: p=0 o p=1)
# 3. La entropía crece más rápido cerca de 0.5 (más sensible a cambios)
# 4. Gini tiene máximo en 0.5, entropía en 1.0 (escalas diferentes)
# 5. En la práctica, ambas dan splits muy similares


El árbol busca el split que **más reduce la impureza** ponderando por el tamaño de los hijos. Eso se llama **ganancia de información** y es lo que decide qué variable y qué corte usar en cada nodo.

### Ejemplo numérico

Supongamos un nodo con 100 observaciones: 60 clase 0, 40 clase 1.

**Impureza del padre:**
$$G_{\text{padre}} = 1 - 0.6^2 - 0.4^2 = 1 - 0.36 - 0.16 = 0.48$$

**Split candidato:** `Credit_Score <= 650`
- Hijo izquierdo (70 obs): 20 clase 0, 50 clase 1 → $G_{\text{izq}} = 1 - (20/70)^2 - (50/70)^2 = 0.41$
- Hijo derecho (30 obs): 40 clase 0, 10 clase 1 → $G_{\text{der}} = 1 - (40/30)^2 - (10/30)^2 = 0.44$

**Impureza ponderada de los hijos:**
$$G_{\text{hijos}} = \frac{70}{100} \cdot 0.41 + \frac{30}{100} \cdot 0.44 = 0.287 + 0.132 = 0.419$$

**Ganancia de información:**
$$\text{Ganancia} = 0.48 - 0.419 = 0.061$$

El árbol prueba **todos los splits posibles** (todas las variables, todos los puntos de corte) y elige el que tenga mayor ganancia.

> ⚠️ **Importante:** El algoritmo es **greedy** (voraz) — elige el mejor split en cada paso sin mirar hacia adelante. No garantiza encontrar el árbol óptimo global, pero es computacionalmente eficiente.

## 3. Logística vs árbol — comparación detallada

| Aspecto | Regresión Logística | Árbol de Decisión |
|---------|-------------------|-------------------|
| **Frontera de decisión** | Lineal (hiperplano) | Escalonada (regiones rectangulares) |
| **No linealidades / interacciones** | Requieren ingeniería manual (e.g., $x_1 \cdot x_2$, $x^2$) | Capturadas automáticamente por los splits |
| **Salida** | Probabilidades calibradas (via función sigmoide) | Proporciones por hoja (pueden no estar calibradas) |
| **Interpretabilidad** | Coeficientes con significado estadístico (log-odds) | Reglas if/else legibles, fácil de explicar |
| **Sensibilidad a outliers** | Media (afectan los coeficientes) | Baja (solo afectan el split, no la magnitud) |
| **Escala de features** | Sensible — necesita estandarización | Indiferente — solo importa el orden relativo |
| **Riesgo de overfitting** | Bajo (especialmente con regularización L2) | **Alto sin poda** — puede memorizar ruido |
| **Extrapolación** | Sí — puede predecir fuera del rango de entrenamiento | **No** — predice la clase de la hoja más cercana |
| **Estabilidad** | Alta — pequeños cambios en datos → pequeños cambios en coeficientes | **Baja** — pequeños cambios pueden cambiar todo el árbol |
| **Calibración de probabilidades** | Buena (sigmoide es una función de calibración) | Regular (proporciones crudas de la hoja) |

### Cuándo usar cada uno

**Usa regresión logística si:**
- La frontera de decisión es aproximadamente lineal
- Necesitas probabilidades bien calibradas (e.g., scoring crediticio regulado)
- Quieres inferencia estadística (p-valores, intervalos de confianza)
- Tienes pocas observaciones
- Priorizas estabilidad del modelo

**Usa árbol de decisión si:**
- La frontera de decisión es claramente no lineal
- Hay interacciones complejas entre variables
- Priorizas interpretabilidad sobre precisión estadística
- Tienes suficientes datos para evitar overfitting
- No necesitas probabilidades calibradas (o puedes calibrarlas después)
- Quieres explicar decisiones individuales con reglas

> 💡 **En la práctica:** Los árboles individuales rara vez se usan en producción — se prefieren **ensembles** (Random Forest, Gradient Boosting) que promedian muchos árboles para reducir la varianza y mejorar la calibración.

Visualmente, en un mismo problema 2D la logística traza una **recta** (frontera lineal) y el árbol traza **rectángulos** (fronteras escalonadas):

### Dataset sintético: "two moons"

Usamos un dataset con forma de dos lunas entrelazadas — un caso clásico donde la frontera de decisión óptima es **no lineal**. Esto permite ver claramente las limitaciones de la regresión logística y las ventajas del árbol.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Generamos un dataset sintético con forma de dos lunas entrelazadas
# n_samples=400: 400 observaciones (200 por clase)
# noise=0.25: ruido gaussiano para hacerlo realista (no perfectamente separable)
# random_state=0: semilla fija para reproducibilidad
Xs, ys = make_moons(n_samples=400, noise=0.25, random_state=0)

# Entrenamos dos modelos sobre los mismos datos
modelos = [
    ('Regresión logística', LogisticRegression().fit(Xs, ys)),
    ('Árbol (max_depth=4)', DecisionTreeClassifier(max_depth=4, random_state=0).fit(Xs, ys)),
]

# Creamos una malla (grid) de puntos para visualizar la frontera de decisión
# linspace genera 300 puntos equidistantes en cada eje
# Añadimos margen de 0.5 para ver mejor los bordes
xx, yy = np.meshgrid(
    np.linspace(Xs[:,0].min()-0.5, Xs[:,0].max()+0.5, 300),
    np.linspace(Xs[:,1].min()-0.5, Xs[:,1].max()+0.5, 300),
)

# Aplanamos la malla en un array 2D de shape (90000, 2) para predecir
grid = np.c_[xx.ravel(), yy.ravel()]

# --- Gráfico comparativo ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (nombre, m) in zip(axes, modelos):
    # Predecimos la clase para cada punto de la malla
    zz = m.predict(grid).reshape(xx.shape)
    
    # contourf dibuja las regiones de decisión con colores
    # levels=[-0.5, 0.5, 1.5] → dos regiones (clase 0 y clase 1)
    # colors: naranja claro para clase 0, azul claro para clase 1
    ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5], colors=['#FFE4B5', '#B0E0E6'], alpha=0.8)
    
    # Scatter de los datos reales
    # c=ys: color según la clase verdadera
    # cmap='coolwarm': rojo para clase 0, azul para clase 1
    ax.scatter(Xs[:,0], Xs[:,1], c=ys, cmap='coolwarm', edgecolor='k', s=25)
    
    ax.set_title(nombre)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

# OBSERVACIONES:
# - Regresión logística: frontera LINEAL (recta) — no puede capturar la forma de luna
#   Muchos puntos mal clasificados en la zona de entrelazamiento
# - Árbol: frontera ESCALONADA (rectángulos) — se adapta mejor a la forma curva
#   Menos errores, especialmente en la zona compleja
#
# CONCLUSIÓN: En problemas con fronteras no lineales, el árbol es superior


## 4. Caso práctico — Loan Default Dataset (Kaggle)

**Dataset:** https://www.kaggle.com/datasets/yasserh/loan-default-dataset  
**Archivo:** `Loan_Default.csv` (148.670 filas × 34 columnas).

### Contexto del problema

Este es un dataset de **scoring crediticio** — uno de los casos de uso más importantes de ML en finanzas. Cada fila representa un préstamo hipotecario y queremos predecir si el prestatario entrará en **default** (incumplimiento de pago).

**¿Por qué es importante?**
- **Bancos:** Necesitan estimar el riesgo para decidir aprobaciones, tasas de interés y reservas de capital (regulación Basel III).
- **Fintechs:** Automatizan decisiones de crédito en tiempo real.
- **Regulación:** Exige explicabilidad de las decisiones (por eso los árboles son populares).

### Estructura del dataset

Las columnas se pueden agrupar así:

#### 1. Identificador / metadata
- `ID`: identificador único del préstamo
- `year`: año de originación

#### 2. Información del solicitante
- `Gender`: género (Male / Female / Sex Not Available / Joint)
- `age`: rango de edad (25-34, 35-44, 45-54, 55-64, 65-74, <25, >74)
- `income`: ingreso mensual (USD)
- `Credit_Score`: puntaje crediticio (300-850, estilo FICO)
- `credit_type`: bureau de crédito (CIB / CRIF / EXP / EQUI)
- `co-applicant_credit_type`: tipo de crédito del co-solicitante

#### 3. Características del préstamo
- `loan_amount`: monto del préstamo (USD)
- `loan_type`: tipo de préstamo (type1 / type2 / type3)
- `loan_purpose`: propósito (p1 / p2 / p3 / p4)
- `loan_limit`: límite de crédito (cf / ncf)
- `Credit_Worthiness`: solvencia crediticia (l1 / l2)
- `term`: plazo en meses (típicamente 180, 240, 360)
- `rate_of_interest`: tasa de interés anual (%)
- `Interest_rate_spread`: diferencia con tasa de referencia
- `Upfront_charges`: cargos iniciales (USD)
- `business_or_commercial`: préstamo comercial (b/c / nob/c)
- `open_credit`: línea de crédito abierta
- `interest_only`: solo paga intereses (no amortiza capital)
- `Neg_ammortization`: amortización negativa (deuda crece)
- `lump_sum_payment`: pago único al final
- `submission_of_application`: tipo de solicitud (to_inst / not_inst)
- `approv_in_adv`: aprobación previa (pre / nopre)

#### 4. Garantía / inmueble
- `property_value`: valor de la propiedad (USD)
- `LTV`: loan-to-value ratio (%) — `loan_amount / property_value * 100`
- `construction_type`: tipo de construcción
- `occupancy_type`: tipo de ocupación (pr = primary residence, sr = second residence, ir = investment)
- `Secured_by`: garantizado por (Home / Land)
- `Security_Type`: tipo de garantía (Direct / Indirect)
- `total_units`: número de unidades en la propiedad (1U / 2U / 3U / 4U)

#### 5. Geografía
- `Region`: región geográfica (North / south / central / North-East)

#### 6. Capacidad de pago
- `dtir1`: debt-to-income ratio (%) — `pagos_mensuales_deuda / ingreso * 100`
  - Métrica clave: mide qué % del ingreso se va en pagar deudas
  - Típicamente, bancos rechazan si dtir1 > 43%

#### 7. Variable objetivo
- `Status`: 1 = entró en default, 0 = pagó al día
- **Tasa de default:** ≈ 24.6% (dataset desbalanceado)

### Objetivo del modelo

**Clasificación binaria:** Predecir `Status` (0 o 1) dado el perfil del préstamo y del solicitante.

**Uso en producción:**
- Aprobar/rechazar solicitudes automáticamente
- Ajustar tasas de interés según el riesgo
- Estimar reservas de capital (provisiones para pérdidas esperadas)
- Cumplir con regulación (explicar decisiones)

In [ ]:
from pathlib import Path
import pandas as pd

# Definimos la ruta al CSV usando pathlib (más robusto que strings de ruta)
DATA = Path('../data/Loan_Default.csv')

# Verificamos que el archivo existe antes de intentar cargarlo
if not DATA.exists():
    raise FileNotFoundError(
        f'No se encontró {DATA}. Descarga el dataset desde '
        'https://www.kaggle.com/datasets/yasserh/loan-default-dataset '
        'y colócalo en data/Loan_Default.csv (ver README.md).'
    )

# Cargamos el CSV completo en un DataFrame de pandas
# Este dataset es grande: 148,670 filas × 34 columnas
df = pd.read_csv(DATA)

# Calculamos la tasa de default (proporción de Status=1)
# mean() sobre una columna binaria da la proporción de 1s
tasa_default = df['Status'].mean()

print('Shape:', df.shape, '| Tasa de default:', round(tasa_default, 3))

# Mostramos las primeras 5 filas para inspeccionar la estructura
df.head()

# OBSERVACIÓN:
# Tasa de default ≈ 24.6% → dataset desbalanceado (75% clase 0, 25% clase 1)
# Esto es típico en scoring crediticio (la mayoría de los préstamos se pagan)
# Implicación: accuracy no es una buena métrica (un modelo que siempre predice 0
# tendría 75% accuracy pero sería inútil). Usaremos F1, ROC-AUC, etc.


In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, classification_report,
)

# Configuramos el estilo visual para todos los gráficos
sns.set_theme(style='whitegrid')

# ─────────────────────────────────────────────────────────────────
# Selección de features
# ─────────────────────────────────────────────────────────────────
# Tomamos un subconjunto representativo (no todas las 34 columnas)
# para que el notebook sea más manejable y el árbol visualizable

# Features numéricas (8 variables)
num_cols = ['loan_amount',      # monto del préstamo
            'rate_of_interest',  # tasa de interés
            'term',              # plazo en meses
            'property_value',    # valor de la propiedad
            'income',            # ingreso mensual
            'Credit_Score',      # puntaje crediticio (300-850)
            'LTV',               # loan-to-value ratio (%)
            'dtir1']             # debt-to-income ratio (%)

# Features categóricas (9 variables)
cat_cols = ['loan_type',                # tipo de préstamo
            'loan_purpose',             # propósito del préstamo
            'Credit_Worthiness',        # solvencia crediticia
            'business_or_commercial',   # préstamo comercial o no
            'occupancy_type',           # tipo de ocupación
            'credit_type',              # bureau de crédito
            'age',                      # rango de edad
            'Region',                   # región geográfica
            'Gender']                   # género

# ─────────────────────────────────────────────────────────────────
# Imputación de valores faltantes
# ─────────────────────────────────────────────────────────────────
# IMPORTANTE: En producción real, calcularías la mediana/moda SOLO del train
# y la aplicarías al test. Aquí lo hacemos sobre todo el dataset por simplicidad.

# Numéricas: rellenamos con la mediana (robusta a outliers)
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())

# Categóricas: rellenamos con el literal 'Unknown'
# Esto crea una categoría explícita para valores faltantes
for c in cat_cols:
    df[c] = df[c].fillna('Unknown')

# ─────────────────────────────────────────────────────────────────
# One-Hot Encoding de categóricas
# ─────────────────────────────────────────────────────────────────
# get_dummies convierte cada categoría en una columna binaria (0/1)
# drop_first=True elimina la primera categoría de cada variable para evitar multicolinealidad
# (solo importa para regresión logística, no para árboles, pero lo hacemos por consistencia)

X = pd.get_dummies(df[num_cols + cat_cols], columns=cat_cols, drop_first=True)
y = df['Status']

print('Shape de X:', X.shape)

# RESULTADO:
# X tiene 148,670 filas y ~50 columnas (8 numéricas + ~42 dummies de las categóricas)
# y es un vector binario (0 o 1)


In [ ]:
# Dividimos en train (80%) y test (20%)
# test_size=0.2 → 20% para test
# random_state=42 → semilla fija para reproducibilidad
# stratify=y → IMPORTANTE: mantiene la misma proporción de clases en train y test
#              Sin stratify, podríamos tener 23% default en train y 26% en test (sesgo)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Mostramos los tamaños resultantes
X_tr.shape, X_te.shape

# RESULTADO:
# Train: 118,936 observaciones (80%)
# Test:  29,734 observaciones (20%)
# Ambos conjuntos tienen ~24.6% de default (gracias a stratify)


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Entrenamiento del árbol de decisión
# ─────────────────────────────────────────────────────────────────
# criterion='gini': usa el índice de Gini para medir impureza (default)
# max_depth=4: árbol poco profundo (máximo 2^4 = 16 hojas)
#              Esto evita overfitting y hace el árbol visualizable
# random_state=42: semilla fija para reproducibilidad
tree = DecisionTreeClassifier(
    criterion='gini',
    max_depth=4,
    random_state=42,
).fit(X_tr, y_tr)

# ─────────────────────────────────────────────────────────────────
# Predicciones sobre el conjunto de test
# ─────────────────────────────────────────────────────────────────
# predict() devuelve la clase predicha (0 o 1)
y_pred = tree.predict(X_te)

# predict_proba() devuelve las probabilidades de cada clase
# [:, 1] extrae la probabilidad de la clase 1 (default)
y_proba = tree.predict_proba(X_te)[:, 1]

# ─────────────────────────────────────────────────────────────────
# Evaluación con múltiples métricas
# ─────────────────────────────────────────────────────────────────
# En clasificación desbalanceada, NO basta con accuracy
# Necesitamos ver precision, recall, F1 y ROC-AUC

# Accuracy: proporción de predicciones correctas (TP + TN) / total
# Problema: puede ser alta aunque el modelo no detecte defaults
print(f'Accuracy  : {accuracy_score(y_te, y_pred):.3f}')

# Precision: de los que predijimos como default, ¿cuántos realmente lo son?
# Precision = TP / (TP + FP)
# Alta precision → pocos falsos positivos (no rechazamos buenos pagadores)
print(f'Precision : {precision_score(y_te, y_pred):.3f}')

# Recall (sensibilidad): de los que realmente son default, ¿cuántos detectamos?
# Recall = TP / (TP + FN)
# Alto recall → pocos falsos negativos (no aprobamos malos pagadores)
print(f'Recall    : {recall_score(y_te, y_pred):.3f}')

# F1: media armónica de precision y recall
# F1 = 2 * (precision * recall) / (precision + recall)
# Balancea ambas métricas — útil cuando las clases están desbalanceadas
print(f'F1        : {f1_score(y_te, y_pred):.3f}')

# ROC-AUC: área bajo la curva ROC (Receiver Operating Characteristic)
# Mide qué tan bien el modelo separa las clases usando las probabilidades
# AUC = 0.5 → modelo aleatorio, AUC = 1.0 → modelo perfecto
# AUC > 0.7 → modelo aceptable, AUC > 0.8 → modelo bueno
print(f'ROC AUC   : {roc_auc_score(y_te, y_proba):.3f}')

# classification_report muestra precision, recall y F1 por clase
# También muestra macro avg (promedio simple) y weighted avg (ponderado por tamaño)
print('\n', classification_report(y_te, y_pred, target_names=['Al día', 'Default']))

# INTERPRETACIÓN:
# - Accuracy ≈ 0.76 → 76% de predicciones correctas (pero puede ser engañoso)
# - Precision ≈ 0.60 → de los que predijimos default, 60% realmente lo son
# - Recall ≈ 0.50 → de los que realmente son default, detectamos 50%
# - F1 ≈ 0.55 → balance entre precision y recall
# - ROC-AUC ≈ 0.72 → modelo aceptable (mejor que aleatorio)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# ─────────────────────────────────────────────────────────────────
# Gráfico 1: Matriz de confusión
# ─────────────────────────────────────────────────────────────────
# Muestra los 4 casos posibles:
# - TN (True Negative): predijimos "al día" y era "al día" (correcto)
# - FP (False Positive): predijimos "default" pero era "al día" (error tipo I)
# - FN (False Negative): predijimos "al día" pero era "default" (error tipo II)
# - TP (True Positive): predijimos "default" y era "default" (correcto)

ConfusionMatrixDisplay(
    confusion_matrix(y_te, y_pred),
    display_labels=['Al día', 'Default'],
).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Matriz de confusión — árbol')

# LECTURA:
# Diagonal principal (TN y TP): predicciones correctas
# Diagonal secundaria (FP y FN): errores
# 
# En scoring crediticio:
# - FP (rechazar buen pagador): costo = venta perdida (~$1k-$5k)
# - FN (aprobar mal pagador): costo = pérdida del préstamo (~$50k-$500k)
# → Los FN son MUCHO más costosos → priorizamos recall sobre precision

# ─────────────────────────────────────────────────────────────────
# Gráfico 2: Curva ROC
# ─────────────────────────────────────────────────────────────────
# Muestra el trade-off entre TPR (recall) y FPR (falsos positivos)
# para diferentes umbrales de decisión

# from_predictions calcula la curva ROC automáticamente
RocCurveDisplay.from_predictions(y_te, y_proba, ax=axes[1])

# Línea diagonal: modelo aleatorio (AUC = 0.5)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Aleatorio')

axes[1].set_title('Curva ROC — árbol')
axes[1].legend()

plt.tight_layout()
plt.show()

# INTERPRETACIÓN DE LA CURVA ROC:
# - Cuanto más se aleja de la diagonal (hacia arriba-izquierda), mejor
# - El área bajo la curva (AUC) resume el rendimiento en un solo número
# - AUC = 0.72 → el modelo es mejor que aleatorio, pero hay margen de mejora
# - Podemos elegir un punto en la curva (umbral) según el trade-off deseado


In [ ]:
fig, ax = plt.subplots(figsize=(18, 9))

# plot_tree dibuja el árbol completo con todas sus reglas
# feature_names: nombres de las variables (en lugar de X[0], X[1], etc.)
# class_names: nombres de las clases (['Al día', 'Default'])
# filled=True: colorea los nodos según la clase mayoritaria
#              Naranja → mayoría clase 0 (al día)
#              Azul → mayoría clase 1 (default)
#              Intensidad del color → pureza del nodo
# rounded=True: bordes redondeados (más estético)
# fontsize=8: tamaño de letra (ajustado para que quepa todo)
plot_tree(
    tree,
    feature_names=X.columns,
    class_names=['Al día', 'Default'],
    filled=True,
    rounded=True,
    ax=ax,
    fontsize=8,
)
ax.set_title('Árbol de clasificación sobre Loan Default (max_depth=4)')
plt.show()

# CÓMO LEER EL ÁRBOL:
# Cada nodo muestra:
# - Condición del split (e.g., "dtir1 <= 38.5")
# - gini: índice de Gini (impureza del nodo)
#         0 = nodo puro (todas las obs de la misma clase)
#         0.5 = máxima impureza (50% clase 0, 50% clase 1)
# - samples: número de observaciones que llegaron a ese nodo
# - value: [n_clase_0, n_clase_1] (frecuencias absolutas)
# - class: clase mayoritaria (la que se predice si llega una obs a esa hoja)
#
# Ejemplo de lectura:
# Si un préstamo tiene dtir1 <= 38.5 → va a la izquierda
#   Si además Credit_Score <= 650 → va a la izquierda de nuevo
#     Si además LTV <= 85 → ...
#       Predicción final: "Default" (clase mayoritaria en esa hoja)
#
# El color indica la clase mayoritaria:
# - Naranja → mayoría "al día" (clase 0)
# - Azul → mayoría "default" (clase 1)
# - Intensidad → pureza (más intenso = más puro)


In [ ]:
# --- Importancia de variables ---
# El árbol calcula automáticamente qué tan importante es cada feature
# basándose en cuánto reduce la impureza (Gini) en los splits

# feature_importances_ es un array con la importancia de cada feature
# Suma 1.0 (100%) — es una proporción relativa
# Ordenamos de menor a mayor y tomamos las top 12 para visualizar
imp = (
    pd.Series(tree.feature_importances_, index=X.columns)
    .sort_values()
    .tail(12)  # top 12 más importantes
)

fig, ax = plt.subplots(figsize=(8, 5))

# Gráfico de barras horizontales (más legible para nombres largos)
imp.plot.barh(ax=ax, color='teal')
ax.set_title('Importancia de variables — DecisionTreeClassifier')
ax.set_xlabel('Importancia relativa')
plt.show()

# INTERPRETACIÓN:
# - Importancia = 0 → la variable NO se usó en ningún split del árbol
# - Importancia alta → la variable se usó en splits que redujeron mucho la impureza
#
# En este caso, las variables más importantes son:
# - dtir1 (debt-to-income ratio): capacidad de pago
# - Credit_Score: historial crediticio
# - LTV (loan-to-value): apalancamiento
# - rate_of_interest: costo del crédito
#
# Esto tiene sentido desde el punto de vista de negocio:
# - dtir1 alto → poco margen para pagar → mayor riesgo de default
# - Credit_Score bajo → mal historial → mayor riesgo
# - LTV alto → poco equity → mayor riesgo si el valor de la propiedad baja
#
# IMPORTANTE: La importancia es relativa a ESTE árbol específico
# Si entrenas con otra semilla o max_depth diferente, puede cambiar
# Para importancias más robustas, usa Random Forest (promedia muchos árboles)


In [ ]:
# --- Comparación contra regresión logística sobre el mismo split ---

# ─────────────────────────────────────────────────────────────────
# Entrenamiento de regresión logística
# ─────────────────────────────────────────────────────────────────
# IMPORTANTE: La regresión logística necesita estandarización
# (el árbol NO la necesita — es indiferente a la escala)

# Ajustamos el scaler SOLO con train (evitar data leakage)
scaler = StandardScaler().fit(X_tr)

# Entrenamos la regresión logística con los datos estandarizados
# max_iter=1000: aumentamos las iteraciones para asegurar convergencia
logit = LogisticRegression(max_iter=1000).fit(scaler.transform(X_tr), y_tr)

# Predecimos sobre test (también estandarizado con el scaler del train)
log_pred = logit.predict(scaler.transform(X_te))
log_proba = logit.predict_proba(scaler.transform(X_te))[:, 1]

# ─────────────────────────────────────────────────────────────────
# Comparación de métricas
# ─────────────────────────────────────────────────────────────────
comparativa = pd.DataFrame({
    'modelo':   ['Decision Tree', 'Regresión logística'],
    'accuracy': [accuracy_score(y_te, y_pred), accuracy_score(y_te, log_pred)],
    'f1':       [f1_score(y_te, y_pred),       f1_score(y_te, log_pred)],
    'roc_auc':  [roc_auc_score(y_te, y_proba), roc_auc_score(y_te, log_proba)],
})

comparativa

# INTERPRETACIÓN:
# El árbol supera a la regresión logística en todas las métricas
# (especialmente en F1 y ROC-AUC)
#
# ¿Por qué?
# - El árbol captura interacciones automáticamente
#   Ejemplo: "Si dtir1 > 40 Y Credit_Score < 650 → alto riesgo"
#   La logística necesitaría que creemos manualmente la feature dtir1*Credit_Score
#
# - El árbol captura no linealidades
#   Ejemplo: El efecto de LTV puede ser diferente para préstamos grandes vs pequeños
#   La logística asume un efecto lineal constante
#
# CONCLUSIÓN:
# En este dataset hay clara evidencia de interacciones y no linealidades
# Por eso el árbol es superior (y en producción se usaría Random Forest o XGBoost)


💡 **Observa la diferencia.** En este dataset el árbol gana al lineal por mucho — señal clara de que hay **interacciones y no linealidades** (p. ej. `dtir1` y `LTV` actuando juntas) que la logística no logra capturar sin features manuales.

In [ ]:
# --- Curva de complejidad (max_depth) ---
# Entrenamos árboles con diferentes profundidades para ver el trade-off sesgo-varianza

# Probamos profundidades de 1 a 15
depths = range(1, 16)

# Listas para guardar el accuracy en train y test para cada profundidad
tr, te = [], []

# Para cada profundidad, entrenamos un árbol y evaluamos
for d in depths:
    # Entrenamos un árbol con profundidad d
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_tr, y_tr)
    
    # Calculamos accuracy en train (qué tan bien ajusta los datos de entrenamiento)
    tr.append(accuracy_score(y_tr, m.predict(X_tr)))
    
    # Calculamos accuracy en test (qué tan bien generaliza a datos nuevos)
    te.append(accuracy_score(y_te, m.predict(X_te)))

# --- Gráfico ---
fig, ax = plt.subplots(figsize=(7, 4))

# Línea azul: accuracy en train (siempre aumenta con la profundidad)
ax.plot(depths, tr, marker='o', label='Train accuracy', color='blue')

# Línea naranja: accuracy en test (aumenta hasta cierto punto, luego se estanca o baja)
ax.plot(depths, te, marker='o', label='Test  accuracy', color='orange')

ax.set_xlabel('max_depth')
ax.set_ylabel('accuracy')
ax.set_title('Curva de complejidad — DecisionTreeClassifier')
ax.legend()
ax.grid(True)
plt.show()

# INTERPRETACIÓN:
# - Train accuracy siempre aumenta → el árbol se ajusta mejor a los datos de entrenamiento
# - Test accuracy aumenta al principio (el modelo aprende patrones reales)
# - Test accuracy se estanca o baja después (overfitting — memoriza ruido del train)
#
# PUNTO ÓPTIMO: donde Test accuracy es máximo (balance entre sesgo y varianza)
# En este caso, parece estar alrededor de max_depth = 6-8
#
# Si Train accuracy >> Test accuracy → overfitting (reducir max_depth o usar poda)
# Si Train accuracy ≈ Test accuracy pero ambos bajos → underfitting (aumentar max_depth)
#
# NOTA: En este dataset, el test accuracy se estabiliza rápido (no baja mucho)
# Esto sugiere que el dataset es grande y robusto (148k observaciones)


## 5. Referencias

- Breiman et al. (1984). *Classification and Regression Trees*.
- Quinlan, J. R. (1986). *Induction of Decision Trees*.
- ISLR cap. 8.
- scikit-learn user guide: https://scikit-learn.org/stable/modules/tree.html
- Dataset: https://www.kaggle.com/datasets/yasserh/loan-default-dataset

## Predicción sobre datos nuevos — uso del modelo en producción

Una vez que el modelo está validado en el conjunto de test, queremos usarlo para predecir sobre datos que **no hemos visto**. En la práctica seguimos tres pasos:

1. **Reentrenar con todos los datos disponibles.** Ya hicimos la validación con la partición train/test; ahora aprovechamos el 100% de la información para que el modelo final tenga la mejor estimación posible de los parámetros.
2. **Aplicar exactamente las mismas transformaciones** que durante el entrenamiento (mismas columnas, mismo encoding, misma escala) — un error muy común en producción es desalinear el preprocesamiento.
3. **Persistir el modelo** con `joblib` (o `pickle`) para reutilizarlo sin reentrenar.

In [ ]:
import joblib

# ─────────────────────────────────────────────────────────────────
# PASO 1: Reentrenamos el modelo final con TODOS los datos disponibles
# ─────────────────────────────────────────────────────────────────
# Ya validamos el rendimiento con train/test. Ahora usamos el 100% de los datos
# para que el modelo final tenga la mejor estimación posible.

modelo_final = DecisionTreeClassifier(
    criterion='gini',
    max_depth=4,
    random_state=42,
).fit(X, y)

# ─────────────────────────────────────────────────────────────────
# PASO 2: Creamos un bundle con todo lo necesario para producción
# ─────────────────────────────────────────────────────────────────
# IMPORTANTE: No basta con guardar el modelo
# También necesitamos guardar:
# - La lista de columnas (orden exacto que espera el modelo)
# - Las listas de num_cols y cat_cols (para replicar el preprocesamiento)

bundle = {
    'modelo':    modelo_final,           # el árbol entrenado
    'columnas':  X.columns.tolist(),     # orden exacto de columnas (post one-hot encoding)
    'num_cols':  num_cols,               # columnas numéricas originales
    'cat_cols':  cat_cols,               # columnas categóricas originales
}

# ─────────────────────────────────────────────────────────────────
# PASO 3: Persistimos el bundle a disco
# ─────────────────────────────────────────────────────────────────
joblib.dump(bundle, 'modelo_loan_tree.pkl')
print('Bundle guardado.')

# Para recuperarlo en otro proceso (API, script batch, notebook diferente):
# bundle = joblib.load('modelo_loan_tree.pkl')
# modelo = bundle['modelo']
# columnas = bundle['columnas']
# num_cols = bundle['num_cols']
# cat_cols = bundle['cat_cols']

# IMPORTANTE: En producción real también deberías guardar:
# - Versión del modelo y fecha de entrenamiento
# - Métricas de validación (F1, ROC-AUC, etc.)
# - Distribuciones de las features (para detectar data drift)
# - Umbrales de decisión (e.g., P(default) > 0.3 → rechazar)


### Inferencia individual — una solicitud de préstamo nueva

Igual que en el notebook anterior, usamos `pd.get_dummies` + `reindex` para alinear columnas con el entrenamiento.

In [ ]:
# Definimos una solicitud de préstamo hipotética
# En producción esto vendría de una API (JSON), formulario web, base de datos, etc.
nuevo_credito_raw = pd.DataFrame([{
    # ─── Features numéricas ───
    'loan_amount':              250000,   # préstamo de $250k
    'rate_of_interest':         4.5,      # tasa de interés 4.5% anual
    'term':                     360,      # plazo de 30 años (360 meses)
    'property_value':           350000,   # propiedad valuada en $350k
    'income':                   7200,     # ingreso mensual de $7,200
    'Credit_Score':             720,      # puntaje crediticio bueno (720/850)
    'LTV':                      71.0,     # loan-to-value 71% (250k/350k)
    'dtir1':                    35.0,     # debt-to-income 35% (aceptable, < 43%)
    
    # ─── Features categóricas ───
    'loan_type':                'type1',
    'loan_purpose':             'p3',
    'Credit_Worthiness':        'l1',
    'business_or_commercial':   'nob/c',  # no es préstamo comercial
    'occupancy_type':           'pr',     # primary residence (vivienda principal)
    'credit_type':              'EXP',    # bureau Experian
    'age':                      '35-44',  # rango de edad
    'Region':                   'North',
    'Gender':                   'Male',
}])

# ─────────────────────────────────────────────────────────────────
# PASO CRÍTICO: Replicar el preprocesamiento del entrenamiento
# ─────────────────────────────────────────────────────────────────
# 1. One-hot encoding de las categóricas (mismo drop_first=True)
nuevo = pd.get_dummies(nuevo_credito_raw, columns=cat_cols, drop_first=True)

# 2. Alinear columnas con el entrenamiento usando reindex
#    Si falta una columna dummy (e.g., loan_type_type2), se rellena con 0
#    Si sobra una columna, se elimina
nuevo = nuevo.reindex(columns=X.columns, fill_value=0)

# ─────────────────────────────────────────────────────────────────
# Predicción
# ─────────────────────────────────────────────────────────────────
# predict_proba devuelve [P(clase 0), P(clase 1)]
# [:, 1] extrae P(default)
prob_default = modelo_final.predict_proba(nuevo)[0, 1]

print(f'P(default) = {prob_default:.3f}')

# Decisión con umbral 0.5 (estándar)
print(f'Decisión @ umbral 0.5: {"DENEGAR" if prob_default >= 0.5 else "APROBAR"}')

# Decisión con umbral 0.3 (banco conservador — prioriza evitar defaults)
print(f'Decisión @ umbral 0.3: {"DENEGAR" if prob_default >= 0.3 else "APROBAR"}  '
      f'(banco conservador)')

# INTERPRETACIÓN:
# - Si P(default) = 0.15 → riesgo bajo → aprobar con tasa estándar
# - Si P(default) = 0.25 → riesgo medio → aprobar con tasa más alta o pedir aval
# - Si P(default) = 0.45 → riesgo alto → rechazar o revisar manualmente
#
# El umbral NO es fijo — depende del apetito de riesgo del banco y del costo de cada error


### Caso de uso: política de aprobación basada en riesgo

Un banco rara vez aprueba/rechaza con un umbral fijo. Lo típico es:

- `P(default) < 0.10` → **aprobación automática** con la tasa estándar.
- `0.10 ≤ P(default) < 0.30` → aprobar pero **subir la tasa** o pedir aval.
- `P(default) ≥ 0.30` → **revisión manual** o rechazo.

El modelo ordena las solicitudes; las reglas de negocio definen los umbrales.

In [ ]:
# Calculamos las probabilidades de default para todo el conjunto de test
test_proba = modelo_final.predict_proba(X_te)[:, 1]

# ─────────────────────────────────────────────────────────────────
# Definimos una política de aprobación basada en riesgo
# ─────────────────────────────────────────────────────────────────
# En la práctica, los bancos NO usan un umbral fijo (0.5)
# Definen políticas con múltiples niveles de riesgo

def politica(p):
    """
    Política de aprobación basada en la probabilidad de default.
    
    Parámetros:
    - p: probabilidad de default (0 a 1)
    
    Retorna:
    - 'auto-aprobado': bajo riesgo → aprobar automáticamente con tasa estándar
    - 'aprobado con condiciones': riesgo medio → aprobar pero con tasa más alta o pedir aval
    - 'revisión manual / rechazo': alto riesgo → revisar manualmente o rechazar
    """
    if p < 0.10:
        return 'auto-aprobado'
    if p < 0.30:
        return 'aprobado con condiciones'
    return 'revisión manual / rechazo'

# Aplicamos la política a todas las observaciones del test
decisiones = pd.DataFrame({
    'P_default': test_proba.round(3),
    'real':      y_te.values,  # clase real (0 o 1)
})
decisiones['politica'] = decisiones['P_default'].apply(politica)

# ─────────────────────────────────────────────────────────────────
# Análisis de la política
# ─────────────────────────────────────────────────────────────────
print('Distribución de decisiones sobre el test set:')
print(decisiones['politica'].value_counts().to_frame('n'))

print('\nTasa de default REAL por categoría de la política:')
print(decisiones.groupby('politica')['real'].mean().round(3).to_frame('tasa_default'))

# INTERPRETACIÓN:
# - 'auto-aprobado': debería tener tasa de default < 10% (bajo riesgo)
# - 'aprobado con condiciones': tasa de default entre 10-30% (riesgo medio)
# - 'revisión manual / rechazo': tasa de default > 30% (alto riesgo)
#
# Si la tasa real coincide con la esperada → el modelo está bien calibrado
# Si la tasa real es mucho mayor → el modelo subestima el riesgo (peligroso)
# Si la tasa real es mucho menor → el modelo sobreestima el riesgo (perdemos ventas)
#
# VENTAJA DEL ÁRBOL:
# Podemos explicar POR QUÉ una solicitud cayó en cada categoría
# (qué reglas se aplicaron) → cumplimiento regulatorio


**Lecciones para producción:**
- En scoring crediticio el desbalance de costos es brutal: un falso negativo (prestar a alguien que va a default) puede costar miles de USD; un falso positivo (rechazar a un buen pagador) cuesta una venta perdida. **Ajustar el umbral** y **revisar la política** son tareas continuas, no decisiones de una sola vez.
- Los árboles dan reglas explicables — útil para cumplir con regulación que exige justificar la decisión al solicitante.
- En la práctica, un solo árbol se reemplaza por un **Random Forest** o **Gradient Boosting** (XGBoost / LightGBM): más AUC y aún explicable con SHAP.

## 6. Carga del modelo desde disco — flujo real de producción

En los ejemplos anteriores todo estaba **en memoria** (mismo notebook). En producción el escenario es diferente:

- El modelo se entrenó en un job separado (o en otro servidor) y se guardó como `.pkl`.
- El servicio de inferencia (una API, un script batch, un container) **solo tiene el `.pkl`** — no tiene acceso a los datos de entrenamiento ni a las variables del notebook.

### Flujo real en producción (scoring crediticio)

```
[Entrenamiento offline]          [Producción / API en tiempo real]
  train.py  ──pkl──►  api.py
                       │
                       1. joblib.load('modelo.pkl')
                       2. recibir solicitud (JSON desde formulario web)
                       3. validar features + replicar preprocesamiento
                       4. modelo.predict_proba(X_nuevo)
                       5. aplicar política de aprobación
                       6. (opcional) extraer reglas del árbol para explicar
                       7. retornar decisión + probabilidad + explicación
```

### Ventajas del árbol en producción

1. **Explicabilidad:** Puedes extraer las reglas que se aplicaron (cumplimiento regulatorio).
2. **No necesita escala:** A diferencia de la regresión logística, no necesitas guardar un `StandardScaler`.
3. **Robusto a outliers:** Un valor extremo en una feature no rompe el modelo.
4. **Rápido:** La inferencia es muy rápida (solo recorrer el árbol, sin multiplicaciones matriciales).

> 🔑 **En producción real:** Se usa Random Forest o XGBoost (ensembles de árboles) en lugar de un solo árbol, pero el flujo es el mismo.

In [ ]:
import joblib
import pandas as pd
from sklearn.tree import export_text

# ─────────────────────────────────────────────────────────────────
# PASO 1: Cargar el bundle desde disco
# Esto es lo ÚNICO que necesitas en producción — sin datos de train,
# sin variables previas del notebook.
# ─────────────────────────────────────────────────────────────────
bundle = joblib.load('modelo_loan_tree.pkl')

modelo = bundle['modelo']
columnas = bundle['columnas']
num_cols = bundle['num_cols']
cat_cols = bundle['cat_cols']

print('Bundle cargado:')
print('- Modelo:', type(modelo).__name__)
print('- Profundidad del árbol:', modelo.get_depth())
print('- Número de hojas:', modelo.get_n_leaves())
print('- Número de features:', len(columnas))

# ─────────────────────────────────────────────────────────────────
# PASO 2: Recibir datos crudos (simulamos una solicitud entrante)
# En producción esto vendría de una API (JSON), formulario web, etc.
# ─────────────────────────────────────────────────────────────────
solicitud_json = {
    # Numéricas
    'loan_amount': 300000,
    'rate_of_interest': 5.2,
    'term': 360,
    'property_value': 400000,
    'income': 8500,
    'Credit_Score': 680,
    'LTV': 75.0,
    'dtir1': 42.0,
    # Categóricas
    'loan_type': 'type2',
    'loan_purpose': 'p1',
    'Credit_Worthiness': 'l2',
    'business_or_commercial': 'nob/c',
    'occupancy_type': 'pr',
    'credit_type': 'CRIF',
    'age': '45-54',
    'Region': 'south',
    'Gender': 'Female',
}

# Convertimos a DataFrame
datos_crudos = pd.DataFrame([solicitud_json])

# ─────────────────────────────────────────────────────────────────
# PASO 3: Preprocesamiento (replicar EXACTAMENTE el del entrenamiento)
# ─────────────────────────────────────────────────────────────────
# 1. Imputación de valores faltantes (si los hay)
for c in num_cols:
    if c in datos_crudos.columns and datos_crudos[c].isna().any():
        # En producción real, usarías la mediana del TRAIN (guardada en el bundle)
        datos_crudos[c] = datos_crudos[c].fillna(0)  # placeholder

for c in cat_cols:
    if c in datos_crudos.columns and datos_crudos[c].isna().any():
        datos_crudos[c] = datos_crudos[c].fillna('Unknown')

# 2. One-hot encoding (mismo drop_first=True)
X_nuevo = pd.get_dummies(datos_crudos[num_cols + cat_cols], columns=cat_cols, drop_first=True)

# 3. Alinear columnas con el entrenamiento
X_nuevo = X_nuevo.reindex(columns=columnas, fill_value=0)

# ─────────────────────────────────────────────────────────────────
# PASO 4: Inferencia + Explicabilidad
# ─────────────────────────────────────────────────────────────────
# Probabilidad de default
prob_default = modelo.predict_proba(X_nuevo)[0, 1]

# Clase predicha
clase = modelo.predict(X_nuevo)[0]

# ID de la hoja donde cayó (útil para debugging y monitoreo)
hoja_id = modelo.apply(X_nuevo)[0]

# Reglas del árbol en formato texto (para explicar la decisión)
reglas = export_text(modelo, feature_names=columnas)

# Política de aprobación
def politica(p):
    if p < 0.10:
        return 'AUTO-APROBADO', 'Tasa estándar'
    if p < 0.30:
        return 'APROBADO CON CONDICIONES', 'Tasa +1.5% o pedir aval'
    return 'REVISIÓN MANUAL / RECHAZO', 'Riesgo alto'

decision, condicion = politica(prob_default)

# ─────────────────────────────────────────────────────────────────
# PASO 5: Respuesta
# ─────────────────────────────────────────────────────────────────
print(f'\n─── RESULTADO ───')
print(f'Probabilidad de default: {prob_default:.1%}')
print(f'Decisión: {decision}')
print(f'Condición: {condicion}')
print(f'Hoja del árbol: #{hoja_id}')
print(f'\nReglas aplicadas (primeras 20 líneas):')
print('\n'.join(reglas.split('\n')[:20]))

# ─────────────────────────────────────────────────────────────────
# BONUS: cómo se vería esto en producción (pseudo-código de una API FastAPI)
# ─────────────────────────────────────────────────────────────────
print("""
─── Ejemplo en una API real (FastAPI) ───────────────────────────

from fastapi import FastAPI
import joblib, pandas as pd
from sklearn.tree import export_text

app = FastAPI()

# Carga UNA vez al iniciar el servidor
bundle = joblib.load("modelo_loan_tree.pkl")
modelo = bundle['modelo']
columnas = bundle['columnas']
num_cols = bundle['num_cols']
cat_cols = bundle['cat_cols']

@app.post("/evaluar_credito")
def evaluar_credito(solicitud: dict):
    # 1. Convertir a DataFrame
    df = pd.DataFrame([solicitud])
    
    # 2. Preprocesamiento
    X = pd.get_dummies(df[num_cols + cat_cols], columns=cat_cols, drop_first=True)
    X = X.reindex(columns=columnas, fill_value=0)
    
    # 3. Predicción
    prob = modelo.predict_proba(X)[0, 1]
    hoja = modelo.apply(X)[0]
    
    # 4. Política
    if prob < 0.10:
        decision = "AUTO-APROBADO"
    elif prob < 0.30:
        decision = "APROBADO CON CONDICIONES"
    else:
        decision = "REVISIÓN MANUAL"
    
    # 5. Explicación (reglas del árbol)
    reglas = export_text(modelo, feature_names=columnas)
    
    return {
        "probabilidad_default": round(float(prob), 3),
        "decision": decision,
        "hoja_id": int(hoja),
        "explicacion": reglas  # reglas legibles para auditoría
    }

──────────────────────────────────────────────────────────────────
""")
